# Evidence-hit F1: pred vs pred0

读取 `results/evidence` 下三类文件并比较：
- `*_base_ev.json`
- `*_pred_ev.json`
- `*_pred0_ev.json`


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('results/evidence')
BASE_SUFFIX = '_base_ev.json'
PRED_SUFFIX = '_pred_ev.json'
PRED0_SUFFIX = '_pred0_ev.json'

VARIANT_SUFFIX = {
    'pred': PRED_SUFFIX,
    'pred0': PRED0_SUFFIX,
}

# pred.general 对应 base 的 General + Applicant
SECTION_MAPPING = {
    'general': ['General', 'Applicant'],
    'proposed_research': ['Proposed research'],
    'training_development': ['Training and development'],
    'sites_support': ['Sites and support'],
    'wpcc': ['Working with people and communities'],
    'application_form': ['Application Form'],
}


In [ ]:
def load_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def parse_pred_obj(d: dict) -> dict:
    out = {}
    for k, v in d.items():
        out[k] = json.loads(v) if isinstance(v, str) else v
    return out


def base_criteria(base_obj: dict, sections: list[str]):
    rows = []
    for sec in sections:
        sec_obj = base_obj.get(sec, {})
        if not isinstance(sec_obj, dict):
            continue
        for crit, crit_obj in sec_obj.items():
            if isinstance(crit_obj, dict) and 'used_chunk_ids' in crit_obj:
                rows.append((sec, crit, set(crit_obj.get('used_chunk_ids') or [])))
    return rows


def pred_criteria(pred_sec_obj: dict):
    rows = []
    for crit, crit_obj in pred_sec_obj.items():
        if isinstance(crit_obj, dict) and 'used_chunk_ids' in crit_obj:
            rows.append((crit, set(crit_obj.get('used_chunk_ids') or [])))
    return rows


def score(base_chunks: set, pred_chunks: set):
    tp = len(base_chunks & pred_chunks)
    fp = len(pred_chunks - base_chunks)
    fn = len(base_chunks - pred_chunks)
    p = tp / (tp + fp) if (tp + fp) else np.nan
    r = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = (2 * p * r / (p + r)) if pd.notna(p) and pd.notna(r) and (p + r) else np.nan
    return tp, fp, fn, p, r, f1


def evaluate_variant(variant: str, suffix: str) -> pd.DataFrame:
    base_paths = {f.name.replace(BASE_SUFFIX, ''): f for f in DATA_DIR.glob(f'*{BASE_SUFFIX}')}
    pred_paths = {f.name.replace(suffix, ''): f for f in DATA_DIR.glob(f'*{suffix}')}
    case_ids = sorted(set(base_paths) & set(pred_paths))

    rows = []
    for cid in case_ids:
        base_obj = load_json(base_paths[cid])
        pred_obj = parse_pred_obj(load_json(pred_paths[cid]))

        for pred_sec, base_secs in SECTION_MAPPING.items():
            b_rows = base_criteria(base_obj, base_secs)
            p_rows = pred_criteria(pred_obj.get(pred_sec, {}))
            n = min(len(b_rows), len(p_rows))

            for i in range(n):
                b_sec, b_crit, b_chunks = b_rows[i]
                p_crit, p_chunks = p_rows[i]
                tp, fp, fn, p, r, f1 = score(b_chunks, p_chunks)
                rows.append({
                    'variant': variant,
                    'case_id': cid,
                    'pred_section': pred_sec,
                    'pair_index': i + 1,
                    'base_section': b_sec,
                    'base_criterion': b_crit,
                    'pred_criterion': p_crit,
                    'tp': tp,
                    'fp': fp,
                    'fn': fn,
                    'precision': p,
                    'recall': r,
                    'f1': f1,
                })

    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> pd.Series:
    tp, fp, fn = df['tp'].sum(), df['fp'].sum(), df['fn'].sum()
    p = tp / (tp + fp) if (tp + fp) else np.nan
    r = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = (2 * p * r / (p + r)) if pd.notna(p) and pd.notna(r) and (p + r) else np.nan
    return pd.Series({
        'criteria_n': len(df),
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'micro_precision': p,
        'micro_recall': r,
        'micro_f1': f1,
        'macro_f1': df['f1'].mean(),
    })


In [ ]:
criterion_df = pd.concat(
    [evaluate_variant(v, s) for v, s in VARIANT_SUFFIX.items()],
    ignore_index=True,
)
print('rows:', len(criterion_df))
criterion_df.head(10)


In [ ]:
overall_summary = (
    criterion_df.groupby('variant', as_index=False)
    .apply(summarize, include_groups=False)
    .sort_values('variant')
    .reset_index(drop=True)
)
overall_summary


In [ ]:
case_summary = (
    criterion_df.groupby(['variant', 'case_id'], as_index=False)
    .apply(summarize, include_groups=False)
    .sort_values(['variant', 'case_id'])
    .reset_index(drop=True)
)
case_summary[['variant','case_id','micro_f1','macro_f1']]


In [ ]:
section_summary = (
    criterion_df.groupby(['variant', 'pred_section'], as_index=False)
    .agg(mean_f1=('f1','mean'))
    .sort_values(['pred_section','variant'])
)
section_summary


In [ ]:
ax = case_summary.pivot(index='case_id', columns='variant', values='micro_f1').plot(kind='bar', figsize=(8,4))
ax.set_title('Case-level Micro F1: pred vs pred0')
ax.set_ylim(0,1)
ax.set_ylabel('Micro F1')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
ax = overall_summary.set_index('variant')[['micro_f1','macro_f1']].plot(kind='bar', figsize=(6,4))
ax.set_title('Overall F1: pred vs pred0')
ax.set_ylim(0,1)
ax.set_ylabel('F1')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
ax = section_summary.pivot(index='pred_section', columns='variant', values='mean_f1').plot(kind='barh', figsize=(9,5))
ax.set_title('Section Mean F1: pred vs pred0')
ax.set_xlim(0,1)
ax.set_xlabel('Mean F1')
plt.tight_layout()
plt.show()


In [ ]:
# pred 相比 pred0 的总体差值
cmp = overall_summary.set_index('variant')[['micro_f1','macro_f1']]
delta = (cmp.loc['pred'] - cmp.loc['pred0']).rename('pred_minus_pred0')
delta.to_frame()
